In [1]:
import os
from dotenv import load_dotenv

from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.tools.retriever import create_retriever_tool
from langchain_classic.agents import create_openai_tools_agent, AgentExecutor

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY is missing. Add it to your .env file.")
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY is missing. Add it to your .env file.")

loader = WebBaseLoader("https://en.wikipedia.org/wiki/Machine_learning")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
documents = splitter.split_documents(docs)

print(f"Loaded {len(docs)} page document(s) and created {len(documents)} chunks.")

embeddings = OpenAIEmbeddings(api_key=openai_api_key)
vectordb = FAISS.from_documents(documents, embeddings)
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

retriever
retriever_tool = create_retriever_tool(
    retriever,
    "machine_learning_search",
    "Search the Wikipedia Machine Learning page for information about machine learning, its methods, applications, and concepts."
)

print(retriever_tool.name)

C:\Users\venka\AppData\Local\Temp\ipykernel_17084\1907854599.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded 1 page document(s) and created 187 chunks.
machine_learning_search


In [2]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=groq_api_key,
    temperature=0
)

llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.5'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000018A959D7A10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000018A947B4830>, model_name='openai/gpt-oss-20b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful research assistant. Use the available tools when they help answer the question. "
        "For questions about machine learning, prefer machine_learning_search. "
        "For research paper questions or arXiv IDs, use the arXiv tool. "
        "If a tool does not provide the answer, be transparent about that."
    ),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

prompt

ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], input_types={'agent_scratchpad': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Ann

In [4]:
import wikipedia

wikipedia.set_user_agent("Machine Learning")

In [5]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)



In [6]:
try:
    result = wiki.run("Machine Learning")
except Exception as e:
    print("Wikipedia tool error:",repr(e))

In [7]:
print(wiki.run("https://en.wikipedia.org/wiki/Machine_learning"))

Page: Wikipedia
Summary: Wikipedia is a free online encyclopedia written and maintained by a community of volunteer editors, known as Wikipedians, through open collaboration and the wiki software MediaWiki. Founded by Jimmy Wales and Larry Sanger in 


In [8]:
import arxiv
from langchain_core.tools import tool

@tool
def arxiv_search(query:str)->str:
    """Search arxiv for academic research paper using a topic or paper ID"""

    try:
        client= arxiv.Client()

        if query.replace(".","").isdigit():
            search = arxiv.Search(id_list=[query],
                                  max_results=1)

        else:
            search=arxiv.Search(
                query=query,
                max_results=3,
                sort_by=arxiv.SortCriterion.Relevance
            )
        papers = list(client.results(search))

        if not papers:
            return "No papers found"

        return "\n\n".join(
            f"Title: {paper.title}\n"
            f"Authors:{','.join(a.name for a in paper.authors)}\n"
            f"Published: {paper.published}\n"
            f"Summary: {paper.summary}\n"
            f"Summary: {paper.entry_id}"
            for paper in papers
        )
    except Exception as e:
        return f"arXiv search failed: {e}"

In [9]:
print(arxiv_search.invoke({"query":"1706.03762"}))

Title: Attention Is All You Need
Authors:Ashish Vaswani,Noam Shazeer,Niki Parmar,Jakob Uszkoreit,Llion Jones,Aidan N. Gomez,Lukasz Kaiser,Illia Polosukhin
Published: 2017-06-12 17:57:34+00:00
Summary: The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task, improving over the existing best results, including ensembles by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model establishes a n

In [10]:
tools = [wiki,arxiv_search,retriever_tool]
for tool in tools:
    print(tool.name)

wikipedia
arxiv_search
machine_learning_search


In [11]:
agent = create_openai_tools_agent(
    llm,tools,prompt
)

In [13]:
agent_executor = AgentExecutor(
    agent=agent,tools=tools,verbose=True
)

In [14]:
response = agent_executor.invoke({
    "input": "What's the paper 1706.03762 about?"
})

print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `arxiv_search` with `{'query': '1706.03762'}`


Title: Attention Is All You Need
Authors:Ashish Vaswani,Noam Shazeer,Niki Parmar,Jakob Uszkoreit,Llion Jones,Aidan N. Gomez,Lukasz Kaiser,Illia Polosukhin
Published: 2017-06-12 17:57:34+00:00
Summary: The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task, improving over the existing best results, including ense

In [15]:
response = agent_executor.invoke({
    "input": "What's the Machine_learning about?"
})

print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `machine_learning_search` with `{'query': 'machine learning'}`


Machine learning - Wikipedia




























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages



















Search











Search






















Appearance
















Donate

Create account

Log in








Personal tools






Donate


Create account


Log in





























Contents
move to sidebar
hide




(Top)





1
History








2
Relationships to other fields




Toggle Relationships to other fields subsection





2.1
Artificial intelligence








2.2
Data compression








2.3
Data mining








2.4
Generalization








2.5
Statistics








2.6
Statistical physics










3
Theory








4
Approaches



